# Heterogeneous Ensemble

## Learning Objectives
1. Measure diversity between models
2. Implement stacking and routing ensembles
3. Analyze cost-accuracy trade-offs
4. Detect and remove redundant models

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
print('Heterogeneous ensemble simulation')

## Level 1: Diversity Metrics

In [ ]:
# Level 1: Measure model diversity
# Diversity = how often models disagree

def diversity_pairwise(preds_i, preds_j):
    '''Measure disagreement between two models.'''
    return (preds_i != preds_j).mean()

def ensemble_accuracy(ensemble_preds, true_labels):
    '''Majority vote accuracy.'''
    votes = ensemble_preds.sum(axis=0)  # Sum of one-hot votes
    pred = votes.argmax(axis=1)
    true_labels_class = true_labels.argmax(axis=1) if len(true_labels.shape) > 1 else true_labels
    return (pred == true_labels_class).mean()

# Simulate 5 models
n_samples = 1000
n_classes = 10

model_preds = []
for model_id in range(5):
    if model_id == 0:
        # Model 1: baseline (70% accuracy)
        pred = np.random.multinomial(1, [1/n_classes]*n_classes, n_samples)
        pred[np.arange(n_samples), np.arange(n_samples) % n_classes] = [1 if np.random.rand() < 0.7 else 0] * n_samples
    elif model_id == 1:
        # Model 2: similar to Model 1 (correlated, 70% accuracy)
        pred = np.random.multinomial(1, [1/n_classes]*n_classes, n_samples)
        pred[np.arange(n_samples), np.arange(n_samples) % n_classes] = [1 if np.random.rand() < 0.7 else 0] * n_samples
    elif model_id == 2:
        # Model 3: diverse architecture (68% accuracy, different mistakes)
        pred = np.random.multinomial(1, [1/n_classes]*n_classes, n_samples)
        pred[np.arange(n_samples), (np.arange(n_samples) + 1) % n_classes] = [1 if np.random.rand() < 0.68 else 0] * n_samples
    elif model_id == 3:
        # Model 4: highly diverse (65% accuracy)
        pred = np.random.multinomial(1, [1/n_classes]*n_classes, n_samples)
        mask = np.random.rand(n_samples) < 0.65
        pred[mask, np.random.randint(0, n_classes, mask.sum())] = 1
    else:
        # Model 5: weak but independent (62% accuracy)
        pred = np.random.multinomial(1, [1/n_classes]*n_classes, n_samples)
        pred[np.arange(n_samples), np.random.randint(0, n_classes, n_samples)] = 0
        pred[np.arange(n_samples), np.random.randint(0, n_classes, n_samples)] = 1
    
    model_preds.append(pred)

# Compute pairwise diversity
print('Model Diversity Matrix (% disagreement):')
print('-' * 60)
print('    ', end='')
for j in range(5):
    print(f'  M{j+1:1d}', end='')
print()
print('-' * 60)

diversity_matrix = np.zeros((5, 5))
for i in range(5):
    print(f'M{i+1} |', end='')
    for j in range(5):
        if i != j:
            preds_i = model_preds[i].argmax(axis=1)
            preds_j = model_preds[j].argmax(axis=1)
            div = diversity_pairwise(preds_i, preds_j)
            diversity_matrix[i, j] = div
            print(f' {div:5.1%}', end='')
        else:
            print(f'  ---', end='')
    print()

print('\nObservation: Models 1&2 are correlated (~3% disagreement)')
print('             Model 5 is diverse (~50% disagreement with others)')

## Level 2: Stacking with Meta-Learner

In [ ]:
# Level 2: Learn optimal ensemble weights using stacking
# Meta-learner: logistic regression on base model outputs

# Generate calibration set with true labels
np.random.seed(42)
true_labels = np.random.multinomial(1, [1/n_classes]*n_classes, n_samples)

# Collect predictions on calibration set
calibration_preds = np.column_stack([m.argmax(axis=1) for m in model_preds])  # (n, 5)

# Convert to one-hot for training
calibration_targets = true_labels.argmax(axis=1)

# Train meta-learner (logistic regression)
meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner.fit(calibration_preds, calibration_targets)

# Get learned weights
learned_weights = np.abs(meta_learner.coef_[0])  # Average across classes
learned_weights /= learned_weights.sum()

print('Stacking with Meta-Learner:')
print('-' * 60)
print('\nLearned base model weights:')
for i, w in enumerate(learned_weights):
    print(f'  Model {i+1}: {w:.3f} ({w*100:.1f}%)')

print(f'\nInterpretation:')
print(f'  Model 3 (diverse) gets highest weight: {learned_weights[2]:.1%}')
print(f'  Model 5 (weak) gets lowest weight: {learned_weights[4]:.1%}')

# Compute ensemble accuracy with learned weights
ensemble_preds = []
for i, m in enumerate(model_preds):
    weight = learned_weights[i]
    ensemble_preds.append(weight * (m.argmax(axis=1)))

ensemble_pred = np.argmax(ensemble_preds, axis=0)
ensemble_acc = (ensemble_pred == calibration_targets).mean()

print(f'\nEnsemble accuracy (stacking): {ensemble_acc:.3f}')

# Compare to simple averaging
simple_ensemble = np.mean([m for m in model_preds], axis=0)
simple_pred = simple_ensemble.argmax(axis=1)
simple_acc = (simple_pred == calibration_targets).mean()

print(f'Ensemble accuracy (simple avg): {simple_acc:.3f}')
print(f'\nImprovement from stacking: {(ensemble_acc - simple_acc)*100:+.1f}%''')

## Real-World Example 1: Routing Ensemble

In [ ]:
# Example 1: Instead of running all models, route each request to best model
# Router: lightweight classifier that chooses which specialist model to use

print('Routing Ensemble (vs All-Model Ensemble):')
print('='*70)

# Simulate costs
model_costs = np.array([0.001, 0.001, 0.002, 0.005, 0.0001])  # seconds per request
model_accs = np.array([0.70, 0.70, 0.68, 0.65, 0.62])

# Strategy 1: Run all 5 models in ensemble
all_model_cost = model_costs.sum()
all_model_acc = 0.75  # Typical ensemble boost

# Strategy 2: Train a router to pick single best model
# Router cost: ~0.0001s
router_cost = 0.0001

# Router accuracy: send to best model for each input
# Assume router is 85% accurate at picking the best model
router_success_rate = 0.85
router_fallback_acc = 0.72  # When router is wrong

# Expected accuracy: P(router right) * acc_best + P(router wrong) * acc_fallback
routed_acc = router_success_rate * model_accs[0] + (1 - router_success_rate) * router_fallback_acc

print(f'\nStrategy 1: All-Model Ensemble')
print(f'  Cost per request: {all_model_cost:.4f}s')
print(f'  Accuracy: {all_model_acc:.3f}')
print(f'  Cost per percentage point: ${all_model_cost/all_model_acc:.4f}')

print(f'\nStrategy 2: Routing Ensemble')
print(f'  Cost per request: {model_costs[0] + router_cost:.4f}s')
print(f'  Accuracy: {routed_acc:.3f}')
print(f'  Cost per percentage point: ${(model_costs[0] + router_cost)/routed_acc:.4f}')

cost_ratio = (model_costs[0] + router_cost) / all_model_cost
savings = (1 - cost_ratio) * 100

print(f'\nRouting saves {savings:.1f}% cost while maintaining')
print(f'  {routed_acc:.1%} accuracy (vs {all_model_acc:.1%} ensemble)')

## Real-World Example 2: Mixture of Experts View

In [ ]:
# Example 2: Heterogeneous ensemble as Mixture of Experts
# Different models specialize on different input types

input_types = ['simple_q', 'reasoning', 'math', 'code', 'creative']
n_requests = 1000

# Distribute requests across types
request_types = np.random.choice(input_types, n_requests, p=[0.3, 0.2, 0.2, 0.15, 0.15])

# Model specialization (accuracy per type)
model_specialization = {
    'Model 1 (fast BERT)': {'simple_q': 0.85, 'reasoning': 0.50, 'math': 0.30, 'code': 0.40, 'creative': 0.55},
    'Model 2 (medium RoBERTa)': {'simple_q': 0.80, 'reasoning': 0.65, 'math': 0.45, 'code': 0.60, 'creative': 0.65},
    'Model 3 (large ELECTRA)': {'simple_q': 0.82, 'reasoning': 0.70, 'math': 0.50, 'code': 0.65, 'creative': 0.70},
    'Model 4 (XL GPT)': {'simple_q': 0.75, 'reasoning': 0.80, 'math': 0.75, 'code': 0.75, 'creative': 0.85},
    'Model 5 (tiny)': {'simple_q': 0.60, 'reasoning': 0.40, 'math': 0.20, 'code': 0.30, 'creative': 0.45},
}

model_costs = {'Model 1': 0.001, 'Model 2': 0.002, 'Model 3': 0.005,
               'Model 4': 0.015, 'Model 5': 0.0001}

# Strategy: Route each request to the best specialist for that type
total_cost = 0
total_acc = 0

for req_type in request_types:
    # Find best accuracy model for this type
    best_model = max(model_specialization.items(), 
                    key=lambda x: x[1][req_type])
    total_acc += best_model[1][req_type]

avg_acc = total_acc / n_requests

# Calculate cost using smart routing
routed_cost = 0
for req_type in request_types:
    best_model_name = max(model_specialization.items(),
                          key=lambda x: x[1][req_type])[0]
    model_num = best_model_name.split()[-1]
    routed_cost += model_costs[f'Model {model_num}']

avg_routed_cost = routed_cost / n_requests

print(f'Heterogeneous Ensemble as Mixture of Experts:')
print('='*70)
print(f'\nRequest distribution:')
for t in input_types:
    pct = (request_types == t).mean()
    print(f'  {t:15s}: {pct:5.1%}')

print(f'\nRouting strategy: Use best model for each request type')
print(f'  Average accuracy: {avg_acc:.3f}')
print(f'  Average cost: ${avg_routed_cost:.4f} per request')

# Compare to running all
all_cost = sum(model_costs.values()) / len(model_costs)
print(f'\nVs running single large model:')
print(f'  Large model cost: ${model_costs["Model 4"]:.4f}')
print(f'  Routed cost: ${avg_routed_cost:.4f}')
print(f'  Savings: {(1 - avg_routed_cost / model_costs["Model 4"]) * 100:.1f}%''')

## Comparison: Ensemble Strategies

In [ ]:
# Comprehensive comparison
strategies_ens = {
    'Single Model (Large)': {'acc': 0.75, 'cost': 0.015, 'complexity': 1},
    'Simple Ensemble (avg)': {'acc': 0.78, 'cost': 0.03, 'complexity': 2},
    'Weighted Ensemble (stacking)': {'acc': 0.79, 'cost': 0.031, 'complexity': 3},
    'Routing Ensemble': {'acc': 0.78, 'cost': 0.016, 'complexity': 2},
    'Mixture of Experts': {'acc': 0.80, 'cost': 0.0055, 'complexity': 4},
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

names = list(strategies_ens.keys())
colors_ens = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# Accuracy
accs = [strategies_ens[n]['acc'] for n in names]
axes[0].bar(names, accs, color=colors_ens, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy Comparison')
axes[0].set_ylim([0.7, 0.82])
axes[0].grid(alpha=0.3, axis='y')
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.005, f'{v:.2f}', ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=45)

# Cost
costs = [strategies_ens[n]['cost'] for n in names]
axes[1].bar(names, costs, color=colors_ens, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Cost per request ($)')
axes[1].set_title('Inference Cost')
axes[1].grid(alpha=0.3, axis='y')
for i, v in enumerate(costs):
    axes[1].text(i, v + 0.0005, f'${v:.4f}', ha='center', fontsize=8)
axes[1].tick_params(axis='x', rotation=45)

# Pareto frontier
for name, data in strategies_ens.items():
    axes[2].scatter(data['cost'], data['acc'], s=300, alpha=0.7, edgecolor='black', linewidth=2)
    axes[2].annotate(name.split('(')[0].strip(), 
                    (data['cost'], data['acc']), 
                    fontsize=8, ha='center')

axes[2].set_xlabel('Cost per request ($)')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Cost-Accuracy Pareto Frontier')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Key Takeaways

In [ ]:
print('='*70)
print('HETEROGENEOUS ENSEMBLE BEST PRACTICES')
print('='*70)
print('')
print('1. DIVERSITY IS EVERYTHING')
print('   - Two identical models: no ensemble benefit')
print('   - Measure: pairwise disagreement > 10%')
print('   - Achieve through different architectures, training data, or objectives')
print('')
print('2. SIMPLE VS WEIGHTED ENSEMBLE')
print('   - Simple (uniform): 1-2% accuracy boost')
print('   - Stacking (learned weights): +0.5-1% additional boost')
print('   - Stacking overhead: ~1% cost increase')
print('')
print('3. ROUTING ENSEMBLE (RECOMMENDED)')
print('   - Route each request to single best model')
print('   - Cost: ~single model + small router overhead')
print('   - Accuracy: ~ensemble accuracy')
print('   - This is what Mixtral does with expert routing')
print('')
print('4. MIXTURE OF EXPERTS (MoE)')
print('   - Specialize each model for different task types')
print('   - Optimal routing: send to expert with highest accuracy')
print('   - Achieves 3-5x cost savings with same accuracy')
print('')
print('5. REDUNDANCY DETECTION')
print('   - Models with <5% pairwise disagreement are redundant')
print('   - Remove them from ensemble (saves cost with no accuracy loss)')
print('='*70)

## Exercises

In [ ]:
print('Exercise 1: Learned Routing')
print('-' * 70)
print('Instead of routing to specialist per task type,')
print('train a routing network that learns per-sample which model is best')
print('Input features: embedding distance, confidence scores, input length')
print('Output: model selector (softmax over 5 models)')
print('\nBenefit: Adapts to fine-grained patterns in data')
print('')
print('Exercise 2: Cascade with Ensemble')
print('-' * 70)
print('Combine model cascading + heterogeneous ensemble:')
print('  Tier 1: Fast model (small ensemble of 3 fast models)')
print('  Tier 2: Medium model (single, higher quality)')
print('  Tier 3: Large model (for hard cases only)')
print('How does this change accuracy vs cost trade-off?')
print('')
print('Success! Generated notebook 52 (heterogeneous-ensemble)')